# Fit3D Dataset → Google Drive
> Downloads Fit3D **Testing Set** + **Dataset Info** directly into Google Drive.
> No local storage needed. Saves to `GYMVISION AI/datasets/fit3d/` in your Drive.

### Run cells top to bottom. Do NOT skip any cell.

In [ ]:
# ── Cell 1: Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

In [ ]:
# ── Cell 2: Paths & URLs ──
import os

# Destination — matches path expected by Kaggle notebook Step 11
FIT3D_DIR  = '/content/drive/MyDrive/GYMVISION AI/datasets/fit3d'
FIT3D_RAW  = f'{FIT3D_DIR}/raw'

for d in [FIT3D_DIR, FIT3D_RAW]:
    os.makedirs(d, exist_ok=True)

# Fit3D download URLs
TESTING_SET_URL  = 'https://fit3d.imar.ro/data/fit3d_test.tar.gz'   # 1.4 GB  ← required
DATASET_INFO_URL = 'https://fit3d.imar.ro/data/fit3d_info.json'     # 12 KB   ← required
TRAINING_SET_URL = 'https://fit3d.imar.ro/data/fit3d_train.tar.gz'  # 18 GB   ← optional

print(f'✅ Folders ready:')
print(f'   {FIT3D_DIR}')
print(f'   {FIT3D_RAW}')

In [ ]:
# ── Cell 3: Download Dataset Info (12 KB) ──
import subprocess

info_path = f'{FIT3D_DIR}/fit3d_info.json'

if os.path.exists(info_path):
    print(f'✅ Dataset Info already exists — skipping')
else:
    print('Downloading Dataset Info (12 KB)...')
    result = subprocess.run(
        ['wget', '-q', '--show-progress', '-O', info_path, DATASET_INFO_URL],
        capture_output=False
    )
    if result.returncode == 0 and os.path.getsize(info_path) > 100:
        print(f'✅ Dataset Info saved → {info_path}')
    else:
        print('❌ Failed — run Cell 7 (cookie auth) instead')

In [ ]:
# ── Cell 4: Download Testing Set (1.4 GB) ──
# Takes ~5–15 min depending on network speed

test_tar = f'{FIT3D_DIR}/fit3d_test.tar.gz'

if os.path.exists(test_tar) and os.path.getsize(test_tar) > 1e8:
    print(f'✅ Testing Set already downloaded — skipping')
else:
    print('Downloading Testing Set (1.4 GB) — please wait...')
    result = subprocess.run(
        ['wget', '-q', '--show-progress', '-O', test_tar, TESTING_SET_URL],
        capture_output=False
    )
    if result.returncode == 0:
        size_gb = os.path.getsize(test_tar) / 1e9
        print(f'✅ Testing Set saved ({size_gb:.2f} GB) → {test_tar}')
    else:
        print('❌ Failed — run Cell 7 (cookie auth) instead')

In [ ]:
# ── Cell 5: Extract Testing Set into raw/ ──
import tarfile

test_tar = f'{FIT3D_DIR}/fit3d_test.tar.gz'

if not os.path.exists(test_tar):
    print('❌ fit3d_test.tar.gz not found — run Cell 4 first')
else:
    print('Extracting Testing Set into raw/ (may take a few minutes)...')
    with tarfile.open(test_tar, 'r:gz') as tar:
        tar.extractall(FIT3D_RAW)
    print(f'✅ Extracted to {FIT3D_RAW}')

In [ ]:
# ── Cell 6: (OPTIONAL) Download Training Set (18 GB) ──
# Only run this if you have enough Drive space and want full angle templates
# Testing Set alone is sufficient for the MAE <= 5 degree thesis gate

DOWNLOAD_TRAINING = False   # ← change to True to download

if DOWNLOAD_TRAINING:
    train_tar = f'{FIT3D_DIR}/fit3d_train.tar.gz'
    if os.path.exists(train_tar) and os.path.getsize(train_tar) > 1e9:
        print('✅ Training Set already downloaded — skipping')
    else:
        print('Downloading Training Set (18 GB) — this will take a long time...')
        result = subprocess.run(
            ['wget', '-q', '--show-progress', '-O', train_tar, TRAINING_SET_URL],
            capture_output=False
        )
        if result.returncode == 0:
            size_gb = os.path.getsize(train_tar) / 1e9
            print(f'✅ Training Set saved ({size_gb:.2f} GB) → {train_tar}')
            print('Extracting...')
            with tarfile.open(train_tar, 'r:gz') as tar:
                tar.extractall(FIT3D_RAW)
            print(f'✅ Extracted to {FIT3D_RAW}')
        else:
            print('❌ Failed — run Cell 7 (cookie auth) instead')
else:
    print('⏭️  Training Set skipped (DOWNLOAD_TRAINING = False)')

## Cell 7 — Only run if Cells 3/4 failed (cookie auth required)

If wget fails, Fit3D requires your browser session cookies:

1. Install **[Get cookies.txt LOCALLY](https://chromewebstore.google.com/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc)** in Chrome
2. Visit `fit3d.imar.ro` while logged in
3. Click the extension → **Export** → save as `fit3d_cookies.txt`
4. Upload `fit3d_cookies.txt` here: Colab left sidebar → Files icon → Upload
5. Then run Cell 7 below

In [ ]:
# ── Cell 7: Cookie-based download (only if Cells 3/4 failed) ──
COOKIE_FILE = '/content/fit3d_cookies.txt'

if not os.path.exists(COOKIE_FILE):
    print('❌ Cookie file not found. Upload fit3d_cookies.txt to Colab first (see instructions above).')
else:
    downloads = [
        ('Dataset Info', DATASET_INFO_URL, f'{FIT3D_DIR}/fit3d_info.json',    False),
        ('Testing Set',  TESTING_SET_URL,  f'{FIT3D_DIR}/fit3d_test.tar.gz',  True),
    ]
    for label, url, out_path, is_tar in downloads:
        if os.path.exists(out_path) and os.path.getsize(out_path) > 100:
            print(f'✅ {label} already exists — skipping')
            continue
        print(f'Downloading {label}...')
        result = subprocess.run(
            ['wget', '-q', '--show-progress', f'--load-cookies={COOKIE_FILE}',
             '--keep-session-cookies', '-O', out_path, url],
            capture_output=False
        )
        if result.returncode == 0:
            size = os.path.getsize(out_path)
            print(f'✅ {label} saved ({size/1e6:.1f} MB)')
            if is_tar:
                print(f'   Extracting into raw/...')
                import tarfile
                with tarfile.open(out_path, 'r:gz') as tar:
                    tar.extractall(FIT3D_RAW)
                print(f'   ✅ Extracted to {FIT3D_RAW}')
        else:
            print(f'❌ {label} failed — contact fit3d.imar.ro support')

In [ ]:
# ── Cell 8: Verify final structure ──
print('🔍 Fit3D folder structure in Drive:')
print(f'   Root: {FIT3D_DIR}\n')

total_files = 0
for root, dirs, files in os.walk(FIT3D_DIR):
    depth = root.replace(FIT3D_DIR, '').count(os.sep)
    if depth > 4:
        continue
    indent = '  ' * depth
    folder = os.path.basename(root) or 'fit3d'
    print(f'{indent}{folder}/')
    shown = 0
    for f in sorted(files):
        if shown < 3:
            print(f'{indent}  {f}')
            shown += 1
    if len(files) > 3:
        print(f'{indent}  ... ({len(files)} files total)')
    total_files += len(files)

print(f'\n✅ Total files: {total_files}')
print('\n🎯 Next step: Run Step 11c in PoseCoach_P01_Kaggle.ipynb')
print('   Upload fit3d/raw/ folder to Kaggle as a private dataset')